# 文件读写与异常处理

---

**学习目标**

1. 掌握 `open()` 的三个关键参数：路径、模式、编码
2. 用 `with` 语句彻底告别"忘了关文件"
3. 分清 `try / except / else / finally` 各自什么时候执行
4. 知道异常是怎么"往上冒"的

**一句话理解**

文件操作就像去图书馆借书：
**打开（open）→ 读或写（read/write）→ 归还（close）**。
不归还的后果是别人借不到（Windows 上文件被占用）。

`with` 语句就是"自动归还服务"：无论中间出什么岔子，它都会帮你把书还回去。


---
## 一、最原始的三步走：open → write → close

> **说明**：本笔记里所有演示用的文件名都带 `demo_` 前缀（`demo_data.txt`、`demo_app.log`），
> 目的是不覆盖你目录里原有的 `data.txt`。你自己练习时可以把前缀去掉。


In [ ]:
# 1) 打开文件。'w' 表示"写入模式"
f = open('demo_data.txt', 'w', encoding='UTF-8')

# 2) 写入内容
f.write('hello world')
f.write('\n第二行内容')

# flush() 把缓冲区里的内容立刻刷到磁盘（不调用的话，close 时也会自动刷）
f.flush()

# 3) 关闭文件 —— 千万别忘
f.close()
print('写完了，去看看 demo_data.txt')


### 关于 `encoding='UTF-8'`：**不要省略**

不写编码，Python 会用操作系统的"默认编码"：

- Windows 中文版：通常是 `GBK`
- Linux / macOS：通常是 `UTF-8`

结果就是**同样的代码在你自己电脑上正常，发给别人就乱码**。
所以涉及中文时，**一律显式写上 `encoding='utf-8'`**。


In [ ]:
# 验证：文件确实被写进去了
f = open('demo_data.txt', 'r', encoding='UTF-8')
print(f.read())
f.close()


---
## 二、文件模式速查表

| 模式 | 含义 | 文件不存在 | 会覆盖原内容吗 |
|---|---|---|---|
| `'r'` | 只读（默认） | **报错** | — |
| `'w'` | 只写 | 自动创建 | **会！清空重写** |
| `'a'` | 追加 | 自动创建 | 不会，写在末尾 |
| `'x'` | 独占创建 | 报错 | 不会，文件已存在会报错 |
| `'r+'` | 读写 | 报错 | 不会 |
| `'w+'` | 读写 | 自动创建 | **会清空** |
| `'rb'` / `'wb'` | 二进制读 / 写 | — | 处理图片、Excel 等非文本文件 |

**最大的坑是 `'w'`**：它会**先清空文件**再写入。
想追加内容必须用 `'a'`。


In [ ]:
# 追加模式演示
with open('demo_data.txt', 'a', encoding='UTF-8') as f:
    f.write('\n这是追加的一行')

## 注意：'a' 模式不会自动加换行，所以要自己写 '\n'

with open('demo_data.txt', 'r', encoding='UTF-8') as f:
    print(f.read())
# 原来的内容 + 新追加的内容，都还在。


### 三种读取方式


In [ ]:
with open('demo_data.txt', 'r', encoding='UTF-8') as f:
    print('=== read() 一次全读成一个大字符串 ===')
    print(repr(f.read()))

with open('demo_data.txt', 'r', encoding='UTF-8') as f:
    print('\n=== readlines() 读成列表，每行一个元素 ===')
    print(f.readlines())

with open('demo_data.txt', 'r', encoding='UTF-8') as f:
    print('\n=== 直接遍历文件对象，逐行读（大文件推荐） ===')
    for i, line in enumerate(f, start=1):
        print(f'  第{i}行：{line.rstrip()}')   # rstrip 去掉行尾换行符


---
## 三、`with` 语句：推荐的标准写法

```python
f = open(...)
try:
    ...
finally:
    f.close()
```

上面这段"手动保证关闭"的代码，`with` 一行就能取代：


In [ ]:
with open('demo_data.txt', 'r', encoding='UTF-8') as f:
    content = f.read()
print('文件内容长度：', len(content))
print('出 with 缩进后，文件已经自动关闭了')
print('还能读吗？', f.closed)         # True —— 已关闭


**`with` 的好处**：即使中间抛异常，文件也**保证**会被关闭。
只要不是必须手动管理文件句柄，**一律用 `with`**。


---
## 四、异常处理：`try / except / else / finally`

这四个块的分工：

| 块 | 什么时候执行 |
|---|---|
| `try` | 放"可能出事"的代码 |
| `except` | 出事时执行（可以写多个，分别对应不同异常） |
| `else` | **没出事**才执行 |
| `finally` | **出不出事都执行**（常用来收尾、关文件） |


In [ ]:
try:
    # 尝试以只读方式打开一个可能不存在的文件
    f = open('demo_not_exist.txt', 'r', encoding='UTF-8')
except FileNotFoundError as e:
    # 捕获"文件不存在"这个具体异常
    print('文件不存在，改用写入模式创建它 ->', e)
    f = open('demo_not_exist.txt', 'w', encoding='UTF-8')
except Exception as e:
    # 兜底：捕获其他所有异常（注意要打印出来，别默默吞掉）
    print(f'其他异常：{type(e).__name__} -> {e}')
    f = None
else:
    # 没出异常才会走到这里
    print('一切正常，没有异常')
finally:
    # 无论走哪条路，这里都会执行
    if f:
        f.close()
    print('finally：收尾完成')


### 4.1 常用异常类型对照表

| 异常 | 什么时候出现 | 典型场景 |
|---|---|---|
| `FileNotFoundError` | 文件不存在 | 读一个没建的文件 |
| `PermissionError` | 没权限 | 写系统目录 |
| `ValueError` | 值不合适 | `int('abc')` |
| `TypeError` | 类型不对 | `'a' + 1` |
| `KeyError` | 字典没有这个键 | `d['不存在']` |
| `IndexError` | 下标越界 | `[1,2][5]` |
| `ZeroDivisionError` | 除以零 | `1 / 0` |

**建议**：能写具体类型就写具体类型，不要一上来就 `except Exception`。
因为异常写得越具体，出了问题越好定位。


In [ ]:
# 演示：同一个 try 里捕获多种不同异常
def safe_div(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return '除数不能为 0'
    except TypeError as e:
        return f'类型不对：{e}'


print(safe_div(10, 2))
print(safe_div(10, 0))
print(safe_div(10, 'x'))


### 4.2 异常会"往上冒"（异常传播）

如果当前函数不处理异常，它会**沿着调用链向上抛**，直到有人接住它，
或者抛到最外层导致程序崩溃。

原始笔记里的例子正是这个机制：


In [ ]:
def func_1():
    print('  func_1 开始执行')
    1 / 0                     # 这里会抛 ZeroDivisionError
    print('  func_1 结束执行')  # 这行永远不会执行


def main():
    try:
        func_1()
    except ZeroDivisionError as e:
        print(f'  main 里接住了异常：{e}')


main()


**流程解读**：

1. `func_1()` 在第 3 行出事，抛出 `ZeroDivisionError`
2. `func_1` 自己没处理，异常往上抛给调用者 `main`
3. `main` 里的 `except` 接住了它，程序**没有崩溃**
4. 因为异常在第 3 行就中断了，所以"func_1 结束执行"永远打不出来

**关键认知**：异常一旦抛出，**当前函数的剩余代码就全部跳过**，
这就是为什么"出错位置之后的 print 都不见了"。


### 4.3 什么时候该用异常处理？

**该用**：

- 和外部世界打交道的地方：读文件、连数据库、调网络接口、解析用户输入
- 这些都是"你可能什么都没做错，但环境会变"的场景

**不该用**：

- 用异常来控制正常的业务流程（比如用 `except` 判断列表是否为空）
- 用 `try: ... except: pass` 把错误藏起来，让问题更难排查

一个反面例子：


In [ ]:
# 反面教材：用 try/except 做本可以用 if 判断的事
data = [1, 2, 3]

# 不好：靠异常来判断下标是否越界
try:
    print(data[10])
except IndexError:
    print('下标越界了（不推荐用异常做流程控制）')

# 更好：先判断再取
if len(data) > 10:
    print(data[10])
else:
    print('长度只有', len(data), '，不去取下标 10（推荐）')


---
## 五、综合实战：一个带异常保护的小日志器

需求：写一个函数，把消息追加到日志文件；如果目录不存在就自动创建。


In [ ]:
import os
from datetime import datetime


def write_log(message, path='demo_app.log'):
    """向日志文件追加一行带时间戳的消息，返回 True/False 表示是否成功"""
    try:
        # 确保目录存在（os.path.dirname 取路径的目录部分）
        folder = os.path.dirname(path)
        if folder and not os.path.exists(folder):
            os.makedirs(folder, exist_ok=True)

        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        with open(path, 'a', encoding='utf-8') as f:
            f.write(f'[{timestamp}] {message}\n')
        return True
    except (PermissionError, OSError) as e:
        print(f'写日志失败：{e}')
        return False


write_log('程序启动')
write_log('用户登录成功')
write_log('这条写到子目录里', path='logs/demo.log')

print('\n--- demo_app.log 的内容 ---')
with open('demo_app.log', 'r', encoding='utf-8') as f:
    print(f.read())


这个例子里，把前面学的都用上了：

- **路径处理**：`os.path.dirname` / `os.makedirs`
- **追加模式**：`'a'` 保证日志不会被覆盖
- **with 语句**：自动关文件
- **异常处理**：磁盘写满、没权限都不至于让程序崩
- **时间戳**：`datetime.now().strftime()`


---
## 六、常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| 用 `'w'` 想追加 | 原内容被清空 | 追加用 `'a'` |
| 忘了 `encoding='utf-8'` | 中文乱码 | 每次 open 都写上 |
| 忘了 `close()` | 内容没落盘 / 文件被占用 | 用 `with` 语句 |
| `read()` 读大文件 | 内存爆掉 | 用 `for line in f` 逐行读 |
| 相对路径找不到文件 | `FileNotFoundError` | 用 `os.path.dirname(__file__)` 拼绝对路径 |
| `except:` 裸捕获 | 把真正的 bug 也吞了 | 写具体异常类型，并打印出来 |
| `except` 里写 `pass` | 出错毫无提示，排查困难 | 至少 `print` 或 `logging` 一下 |
| 把文件对象当字符串用 | `TypeError` | `f.read()` 才拿到内容 |
| `\n` 被 Windows 自动转成 `\r\n` | 跨平台换行不一致 | 用 `newline=''`，或统一按 `\n` 处理 |

### 关于相对路径的实用技巧

相对路径是相对于**运行程序时的当前目录**，不是相对于脚本文件。
所以最稳的写法是"以脚本所在目录为基准"：


In [ ]:
import os

# __file__ 是当前脚本的路径（在 notebook 里是临时路径，所以下面只做示意）
BASE_DIR = os.path.dirname(os.path.abspath('.'))
target = os.path.join(BASE_DIR, 'demo_data.txt')
print('脚本目录：', BASE_DIR)
print('拼出来的路径：', target)
print('更现代的写法（Python 3.4+）：', os.path.join(os.getcwd(), 'demo_data.txt'))


---
## 七、动手练习

**练习 1**：写一个函数 `save_lines(path, lines)`，把一个字符串列表逐行写进文件。
要求用 `with`，并指定 `encoding='utf-8'`。

**练习 2**：读取 `demo_data.txt`，统计它有几行、总共有多少个字符（不含换行）。

**练习 3**：写一个 `safe_read(path)`：文件存在就返回内容，不存在返回 `None`，
并且不抛异常。

**练习 4**：把 `demo_data.txt` 的内容复制到 `data_backup.txt`
（提示：读出来再写进去）。

**练习 5**：写一个"学生成绩"小程序，把用户输入的姓名和分数追加到 `scores.txt`，
然后把它读回来打印成表格。

**练习 6**：解释下面代码为什么不会创建文件，以及该怎么改：
```python
f = open('a.txt', 'w')
f.write('hello')
# 忘了 close
```

---

**答案思路（先自己做）**

```python
# 练习 1
def save_lines(path, lines):
    with open(path, 'w', encoding='utf-8') as f:
        for line in lines:
            f.write(line + '\n')

# 练习 2
with open('demo_data.txt', encoding='utf-8') as f:
    lines = f.readlines()
print('行数：', len(lines))
print('字符数：', sum(len(l.rstrip('\n')) for l in lines))

# 练习 3
def safe_read(path):
    try:
        with open(path, encoding='utf-8') as f:
            return f.read()
    except FileNotFoundError:
        return None
```
